<a href="https://colab.research.google.com/github/JoelForson/ECON5200-Applied-Data-Analytics-in-Economics/blob/main/Minimum%20Wage%20Case%20Study/notebooks/02_Replication_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# Load and Organize Data

In [2]:
col_names = [
    'STORE_ID', 'STATE', 'EMPFT', 'EMPPT', 'NMGRS', 'WAGE_ST',
    'INCTIME', 'FIRSTINC', 'BONUS', 'PCTAFF', 'MEAL',
    'OPEN', 'HRSOPEN', 'PSODA', 'PFRY', 'PENTREE', 'NREGS', 'NREGS11',
    'TYPE', 'STATUS2', 'DATE2', 'NCALLS', 'EMPFT2', 'EMPPT2', 'NMGRS2',
    'WAGE_ST2', 'INCTIME2', 'FIRSTIN2', 'SPECIAL2', 'MEALS2',
    'OPEN2R', 'HRSOPEN2', 'PSODA2', 'PFRY2', 'PENTREE2', 'NREGS2', 'NREGS112'
]
df = pd.read_csv('https://raw.githubusercontent.com/JoelForson/ECON5200-Applied-Data-Analytics-in-Economics/refs/heads/main/Minimum%20Wage%20Case%20Study/data/raw/public.dat', sep=r'\s+', header=None,
                 names=col_names, na_values=['.'])

# FTE = full-time + 0.5×part-time + managers  (Card-Krueger definition)
df['FTE']  = df['EMPFT']  + 0.5 * df['EMPPT']  + df['NMGRS']
df['FTE2'] = df['EMPFT2'] + 0.5 * df['EMPPT2'] + df['NMGRS2']

# Balanced panel: keep only stores that completed Wave 2
df_bal = df[df['STATUS2'] == 1].copy()
print(f"Balanced panel: {len(df_bal)} stores  |  NJ: {df_bal.STATE.sum()}  PA: {(df_bal.STATE==0).sum()}")

Balanced panel: 399 stores  |  NJ: 450  PA: 204


In [3]:
df

,,,,,,,,,STORE_ID,STATE,EMPFT,EMPPT,NMGRS,WAGE_ST,INCTIME,FIRSTINC,BONUS,PCTAFF,...,MEALS2,OPEN2R,HRSOPEN2,PSODA2,PFRY2,PENTREE2,NREGS2,NREGS112,FTE,FTE2
46,1,0,0,0,0,0,1,0,0,0,30.0,15.0,3.0,NaN,19.0,NaN,1,NaN,...,2.0,6.5,16.5,1.03,NaN,0.94,4.0,4.0,40.50,24.00
49,2,0,0,0,0,0,1,0,0,0,6.5,6.5,4.0,NaN,26.0,NaN,0,NaN,...,2.0,10.0,13.0,1.01,0.89,2.35,4.0,4.0,13.75,11.50
506,2,1,0,0,0,0,1,0,0,0,3.0,7.0,2.0,NaN,13.0,0.37,0,30.0,...,1.0,11.0,11.0,0.95,0.74,2.33,4.0,3.0,8.50,10.50
56,4,1,0,0,0,0,1,0,0,0,20.0,20.0,4.0,5.00,26.0,0.10,1,0.0,...,2.0,10.0,12.0,0.92,0.79,0.87,2.0,2.0,34.00,20.00
61,4,1,0,0,0,0,1,0,0,0,6.0,26.0,5.0,5.50,52.0,0.15,1,0.0,...,2.0,10.0,12.0,1.01,0.84,0.95,2.0,2.0,24.00,35.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423,2,1,1,0,0,1,0,0,0,3,2.0,10.0,2.0,4.95,13.0,0.50,1,33.0,...,1.0,11.0,11.0,1.05,0.84,2.32,3.0,2.0,9.00,23.75
424,2,1,1,0,0,1,0,0,0,0,3.5,6.5,3.0,4.75,13.0,0.25,0,45.0,...,1.0,11.0,14.0,1.05,0.94,2.32,5.0,3.0,9.75,17.50
426,3,1,1,0,0,1,0,0,0,0,3.0,33.0,5.0,4.25,26.0,0.15,0,75.0,...,2.0,6.0,18.0,1.11,1.05,1.05,6.0,5.0,24.50,20.50
427,4,0,1,0,0,1,0,0,0,3,7.0,8.0,3.0,4.75,13.0,0.10,0,40.0,...,2.0,10.5,12.5,1.11,1.09,2.07,2.0,2.0,14.00,20.50


# Descriptive Table Reconstruction


In [4]:
# ── 2. Descriptive Table (replicates Table 2) ────────────────
vars_wave1 = {'FTE': 'FTE',  'WAGE_ST': 'WAGE_ST',
              'PSODA': 'PSODA', 'PFRY': 'PFRY', 'PENTREE': 'PENTREE'}
vars_wave2 = {'FTE2': 'FTE', 'WAGE_ST2': 'WAGE_ST',
              'PSODA2': 'PSODA', 'PFRY2': 'PFRY', 'PENTREE2': 'PENTREE'}

def summary_row(series):
    return f"{series.mean():.2f}  ({series.std():.2f})"

rows = []
for w1_col, label in vars_wave1.items():
    w2_col = [k for k, v in vars_wave2.items() if v == label][0]
    rows.append({
        'Variable'      : label,
        'NJ Before'     : summary_row(df_bal.loc[df_bal.STATE==1, w1_col].dropna()),
        'NJ After'      : summary_row(df_bal.loc[df_bal.STATE==1, w2_col].dropna()),
        'PA Before'     : summary_row(df_bal.loc[df_bal.STATE==0, w1_col].dropna()),
        'PA After'      : summary_row(df_bal.loc[df_bal.STATE==0, w2_col].dropna()),
    })

table2 = pd.DataFrame(rows).set_index('Variable')
print("\n=== TABLE 2 RECONSTRUCTION  (mean  (SD)) ===")
print(table2.to_string())


=== TABLE 2 RECONSTRUCTION  (mean  (SD)) ===
              NJ Before       NJ After       PA Before       PA After
Variable                                                             
FTE       21.17  (7.47)  21.25  (7.12)  21.88  (10.64)  21.84  (9.19)
WAGE_ST    4.52  (0.29)   5.03  (0.31)    4.62  (0.35)   4.99  (0.24)
PSODA      1.05  (0.08)   1.04  (0.07)    1.04  (0.09)   1.04  (0.10)
PFRY       0.92  (0.09)   0.95  (0.09)    0.91  (0.11)   0.94  (0.11)
PENTREE    1.21  (0.56)   1.21  (0.55)    1.34  (0.64)   1.34  (0.64)


# 3. Simple Difference-of-Means


In [5]:
nj = df_bal[df_bal.STATE == 1]
pa = df_bal[df_bal.STATE == 0]

nj_before = nj['FTE'].mean()
nj_after  = nj['FTE2'].mean()
pa_before = pa['FTE'].mean()
pa_after  = pa['FTE2'].mean()

diff_nj = nj_after  - nj_before   # NJ change
diff_pa = pa_after  - pa_before   # PA change
did     = diff_nj   - diff_pa     # DiD estimate

print("\n=== SIMPLE DIFFERENCE-OF-MEANS ===")
print(f"  NJ  Before : {nj_before:.3f}   After : {nj_after:.3f}   Δ = {diff_nj:+.3f}")
print(f"  PA  Before : {pa_before:.3f}   After : {pa_after:.3f}   Δ = {diff_pa:+.3f}")
print(f"  DiD (NJ–PA): {did:+.3f}  ← causal estimate")


=== SIMPLE DIFFERENCE-OF-MEANS ===
  NJ  Before : 21.167   After : 21.250   Δ = +0.083
  PA  Before : 21.880   After : 21.836   Δ = -0.044
  DiD (NJ–PA): +0.127  ← causal estimate


 # Regression Implementation, Standard Errors, and Clustering

In [7]:
wave1 = df_bal[['STORE_ID','STATE','FTE','WAGE_ST','PSODA']].copy()
wave1['POST'] = 0

wave2 = df_bal[['STORE_ID','STATE','FTE2','WAGE_ST2','PSODA2']].copy()
wave2.columns = ['STORE_ID','STATE','FTE','WAGE_ST','PSODA']
wave2['POST'] = 1

panel = pd.concat([wave1, wave2], ignore_index=True)
panel = panel.dropna(subset=['FTE']).reset_index(drop=True)

# DiD dummies
panel['TREAT']      = panel['STATE']               # 1 = NJ
panel['TREAT_POST'] = panel['TREAT'] * panel['POST']  # interaction = causal term

print(f"\nLong panel shape: {panel.shape}")
print(panel.head(6))

model = smf.ols('FTE ~ TREAT + POST + TREAT_POST', data=panel)
result   = model.fit(cov_type='cluster', cov_kwds={'groups': panel['STORE_ID']})

print("\n=== DiD REGRESSION  (clustered SEs at store level) ===")
print(result.summary().tables[1])   # coefficient table only — clean output

print(f"\n  ┌───────────────────────────────────────┐")
print(f"  │  DiD Estimate  (β3) : {result.params['TREAT_POST']:+.4f}         │")
print(f"  │  Clustered SE       : {result.bse['TREAT_POST']:.4f}          │")
print(f"  │  t-statistic        : {result.tvalues['TREAT_POST']:.4f}          │")
print(f"  │  p-value            : {result.pvalues['TREAT_POST']:.4f}          │")
print(f"  └───────────────────────────────────────┘")


Long panel shape: (777, 8)
   STORE_ID  STATE    FTE  WAGE_ST  PSODA  POST  TREAT  TREAT_POST
0         0      0  40.50      NaN   1.03     0      0           0
1         0      0  13.75      NaN   1.01     0      0           0
2         0      0   8.50      NaN   0.95     0      0           0
3         0      0  34.00      5.0   0.87     0      0           0
4         0      0  24.00      5.5   0.87     0      0           0
5         0      2  20.50      5.0   0.87     0      2           0

=== DiD REGRESSION  (clustered SEs at store level) ===
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     22.1092      0.008   2889.363      0.000      22.094      22.124
TREAT         -0.8521      0.155     -5.487      0.000      -1.156      -0.548
POST          -0.1213      0.583     -0.208      0.835      -1.264       1.021
TREAT_POST     0.3080      0.263      1.171      0.2

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 1
  warnings.warn('covariance of constraints does not have full '
